# Trainval Archive Selection

Based on the current archive-size and scene-distribution checks, sensor archives **01, 03 and 05 are recommended** for the expanded analysis.

Together, these three archives contain **252 non-overlapping scenes**, meeting the target of approximately 200–300 scenes while keeping the download size relatively small.

The subset retains variation across weather, area, daytime, season, lighting, structure and vehicle-motion conditions.

A limitation is that the subset is selected at the **archive level rather than by individually optimising scenes**, so it is not intended to be a statistically representative sample of the full trainval dataset. Some less common conditions, particularly **roadworks**, are also under-represented.

The code below is used only to verify the archive structure, scene counts, non-overlap and the diversity of the recommended subset.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from remotezip import RemoteZip

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import TRAINVAL_METADATA_ROOT
from src.data_io import load_json
from src.sensor_observation import build_sensor_data_df
from src.annotation_velocity import (
    build_annotation_trajectory_df,
    build_annotation_pairs,
    add_time_differences,
    add_position_changes,
    add_reference_velocity,
    add_scene_tokens,
)

In [2]:
scenes = load_json(TRAINVAL_METADATA_ROOT / "scene.json")
samples = load_json(TRAINVAL_METADATA_ROOT / "sample.json")
annotations = load_json(TRAINVAL_METADATA_ROOT / "sample_annotation.json")
instances = load_json(TRAINVAL_METADATA_ROOT / "instance.json")
categories = load_json(TRAINVAL_METADATA_ROOT / "category.json")

sensor_df = build_sensor_data_df(
    metadata_root=TRAINVAL_METADATA_ROOT
)

print("Total scenes:", len(scenes))
print("Scenes with sensor metadata:", sensor_df["Scene Token"].nunique())

Total scenes: 598
Scenes with sensor metadata: 598


In [3]:
base_url = (
    "https://man-truckscenes.s3.eu-central-1.amazonaws.com/"
    "release/trainval/"
)

file_to_scene = dict(
    zip(
        sensor_df["Filename"].str.replace("\\", "/", regex=False),
        sensor_df["Scene Token"],
    )
)

archive_scene_sets = {}
archive_results = []

for i in range(1, 8):
    archive_name = f"man-truckscenes_sensordata0{i}_v1.2-trainval.zip"

    with RemoteZip(base_url + archive_name) as z:
        filenames = [
            name.replace("man-truckscenes/", "", 1)
            for name in z.namelist()
            if not name.endswith("/")
        ]

    archive_scenes = {
        file_to_scene[name]
        for name in filenames
        if name in file_to_scene
    }

    archive_scene_sets[i] = archive_scenes

    archive_results.append({
        "Archive": f"sensordata0{i}",
        "Scenes": len(archive_scenes),
    })

archive_summary = pd.DataFrame(archive_results)
archive_summary

,Archive,Scenes
0,sensordata01,84
1,sensordata02,84
2,sensordata03,84
3,sensordata04,84
4,sensordata05,84
5,sensordata06,84
6,sensordata07,94


In [4]:
all_archive_scenes = set().union(*archive_scene_sets.values())

total_archive_scene_count = sum(
    len(scenes)
    for scenes in archive_scene_sets.values()
)

print("Unique scenes:", len(all_archive_scenes))
print("Total archive scene count:", total_archive_scene_count)

Unique scenes: 598
Total archive scene count: 598


In [5]:
def parse_scene_description(description):
    values = {}

    if isinstance(description, str):
        for item in description.split(";"):
            if "." in item:
                key, value = item.split(".", 1)
                values[key] = value

    return values


scene_info = pd.DataFrame([
    {
        "Scene Token": scene["token"],
        "Scene Name": scene.get("name"),
        "Description": scene.get("description"),
    }
    for scene in scenes
])

scene_context = (
    scene_info["Description"]
    .apply(parse_scene_description)
    .apply(pd.Series)
)

scene_info = pd.concat(
    [scene_info, scene_context],
    axis=1,
)


trajectory_df = build_annotation_trajectory_df(
    samples,
    annotations,
    instances,
    categories,
)

velocity_df = build_annotation_pairs(trajectory_df)
velocity_df = add_time_differences(velocity_df)
velocity_df = add_position_changes(velocity_df)
velocity_df = add_reference_velocity(velocity_df)
velocity_df = add_scene_tokens(velocity_df, samples)

vehicle_velocity_df = velocity_df[
    velocity_df["category"].str.startswith("vehicle.")
].copy()

scene_vehicle_summary = (
    vehicle_velocity_df
    .groupby("scene_token")
    .agg(
        Num_Vehicles=("instance_token", "nunique"),
        Median_Vehicle_Speed_mps=("speed_2d_mps", "median"),
        P95_Vehicle_Speed_mps=(
            "speed_2d_mps",
            lambda x: x.quantile(0.95),
        ),
    )
    .reset_index()
    .rename(columns={"scene_token": "Scene Token"})
)

scene_summary = scene_info.merge(
    scene_vehicle_summary,
    on="Scene Token",
    how="left",
)

In [6]:
selected_archives = [1, 3, 5]

selected_tokens = set().union(
    *[
        archive_scene_sets[i]
        for i in selected_archives
    ]
)

selected_subset = scene_summary[
    scene_summary["Scene Token"].isin(selected_tokens)
].copy()

print("Selected archives:", selected_archives)
print("Selected scenes:", len(selected_subset))

for column in [
    "weather",
    "area",
    "daytime",
    "season",
    "lighting",
    "structure",
    "construction",
]:
    print(f"\n{column}")
    print(selected_subset[column].value_counts())

print("\nVehicle summary")
display(
    selected_subset[
        [
            "Num_Vehicles",
            "Median_Vehicle_Speed_mps",
            "P95_Vehicle_Speed_mps",
        ]
    ].describe().round(2)
)

Selected archives: [1, 3, 5]
Selected scenes: 252

weather
weather
clear            145
rain              43
overcast          32
snow              21
fog               10
other_weather      1
Name: count, dtype: int64

area
area
highway        180
rural           23
terminal        22
city            18
residential      6
parking          2
other_area       1
Name: count, dtype: int64

daytime
daytime
noon       146
morning     82
evening     24
Name: count, dtype: int64

season
season
autumn    101
summer     91
winter     60
Name: count, dtype: int64

lighting
lighting
illuminated       204
dark               19
twilight           16
glare              12
other_lighting      1
Name: count, dtype: int64

structure
structure
regular      163
underpass     50
overpass      22
bridge        14
tunnel         3
Name: count, dtype: int64

construction
construction
unchanged    250
roadworks      2
Name: count, dtype: int64

Vehicle summary


,Num_Vehicles,Median_Vehicle_Speed_mps,P95_Vehicle_Speed_mps
count,252.00,252.00,252.00
mean,49.98,20.38,30.91
std,28.80,12.24,13.14
min,4.00,0.05,0.16
25%,30.00,7.84,22.33
50%,46.00,25.26,34.41
75%,63.00,29.99,41.10
max,209.00,39.84,57.84
